# Pharma Tokenizer Benchmark - Colab Demo

Japanese pharmaceutical terms often get fragmented by general-purpose tokenizers.

This notebook compares two tokenizers and exports CSV / JSONL reports.

In [ ]:
!pip -q install transformers sentencepiece fugashi unidic-lite tiktoken pandas pyyaml


In [ ]:
import re
import pandas as pd
import tiktoken
from transformers import AutoTokenizer
from google.colab import files

TOK_A = "cl-tohoku/bert-base-japanese-v3"
TOK_B = "cl100k_base"
FRAG_THRESHOLD = 3

drug_strings = [
    "オルプリノン塩酸塩",
    "ロプリノン塩酸塩",
    "ミルリノン",
    "ドブタミン",
    "イバブラジン",
    "アムロジピン",
    "アスピリン",
    "ジアゼパム",
    "メチルプレドニゾロンコハク酸エステルナトリウム",
    "ミルリノン（注）",
    "イブブラジン",
    "アムロディピン",
    "4-[(3,4-dimethoxyphenyl)methyl]-1,2-dihydropyridazin-3-one hydrochloride",
    "methyl (2S)-2-[[4-(dimethylamino)phenyl]sulfonylamino]-3-phenylpropanoate",
]


In [ ]:
tok_a = AutoTokenizer.from_pretrained(TOK_A, use_fast=True)

def encode_a(text):
    ids = tok_a.encode(text, add_special_tokens=False)
    return tok_a.convert_ids_to_tokens(ids)

enc_b = tiktoken.get_encoding(TOK_B)

def encode_b(text):
    ids = enc_b.encode(text)
    tokens = []
    for i in ids:
        try:
            tokens.append(enc_b.decode([i]))
        except Exception:
            tokens.append(f"<{i}>")
    return tokens


In [ ]:
rows = []

for text in drug_strings:
    tokens_a = encode_a(text)
    tokens_b = encode_b(text)
    chars = max(len(text), 1)

    rows.append({
        "input": text,
        "len_A": len(tokens_a),
        "len_B": len(tokens_b),
        "delta_B_minus_A": len(tokens_b) - len(tokens_a),
        "tokens_A": " | ".join(tokens_a),
        "tokens_B": " | ".join(tokens_b),
        "token_per_char_A": len(tokens_a) / chars,
        "token_per_char_B": len(tokens_b) / chars,
        "overfrag_A": len(tokens_a) > FRAG_THRESHOLD,
        "overfrag_B": len(tokens_b) > FRAG_THRESHOLD,
    })

df = pd.DataFrame(rows)
display(df)


In [ ]:
summary = pd.DataFrame([
    {
        "tokenizer": TOK_A,
        "overfrag_count": int(df["overfrag_A"].sum()),
        "total": len(df),
        "overfrag_rate_%": round(df["overfrag_A"].mean() * 100, 1),
        "avg_token_per_char": round(df["token_per_char_A"].mean(), 3),
    },
    {
        "tokenizer": TOK_B,
        "overfrag_count": int(df["overfrag_B"].sum()),
        "total": len(df),
        "overfrag_rate_%": round(df["overfrag_B"].mean() * 100, 1),
        "avg_token_per_char": round(df["token_per_char_B"].mean(), 3),
    },
])

display(summary)

print("BがAより細切れになった上位")
display(df.sort_values("delta_B_minus_A", ascending=False).head(10)[[
    "input", "len_A", "len_B", "delta_B_minus_A", "tokens_A", "tokens_B"
]])


In [ ]:
def is_japanese_like(text):
    return bool(re.search(r"[\u3040-\u30FF\u4E00-\u9FFF]", text))

suggestions = (
    df[(df["overfrag_A"]) | (df["overfrag_B"])]
    ["input"]
    .drop_duplicates()
    .tolist()
)
suggestions = [s for s in suggestions if is_japanese_like(s)]

print("tokenizer.add_tokens([...]) 候補")
for s in suggestions:
    print(s)


In [ ]:
df.to_csv("tokenization_report.csv", index=False, encoding="utf-8-sig")
df.to_json("tokenization_report.jsonl", orient="records", force_ascii=False, lines=True)

with open("add_tokens_candidates.txt", "w", encoding="utf-8") as f:
    for s in suggestions:
        f.write(s + "\n")

files.download("tokenization_report.csv")
files.download("tokenization_report.jsonl")
files.download("add_tokens_candidates.txt")
